## 0_MaxResAndStack.py

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาดดั้งเดิม 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth (ขนาดดั้งเดิม 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(depth_frame).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 # เพิ่มความหนาของตัวอักษรเพราะภาพต้นฉบับใหญ่ขึ้นมาก
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดภาพให้เท่ากันเพื่อแสดงผล (Display Only)
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB (สัดส่วน 16:9 เท่ากัน)
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ---------------------------------------------------------
        # ย่อขนาดสำหรับแสดงบนหน้าจอ (Window Scaling)
        # ---------------------------------------------------------
        # ภาพต่อกันแนวนอนตอนนี้มีขนาด 3840 x 1080 พิกเซล ซึ่งใหญ่กว่าจอคอมพิวเตอร์ทั่วไป
        # เราจึงต้องย่อ "เฉพาะตอนแสดงหน้าต่าง" ลง 50% ให้เหลือ 1920 x 540
        # (ข้อมูลดิบในตัวแปร color_frame และ depth_frame ยังคงเป็นความละเอียดสูงสุด)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Resolution', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27) เพื่อออกจากลูป
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...
⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)
กำลังสตรีมภาพ...
⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว


# 1_RawStream

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense ด้วยความละเอียดสูงสุด...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาดดั้งเดิม 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth (ขนาดดั้งเดิม 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(depth_frame).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 # เพิ่มความหนาของตัวอักษรเพราะภาพต้นฉบับใหญ่ขึ้นมาก
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดภาพให้เท่ากันเพื่อแสดงผล (Display Only)
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB (สัดส่วน 16:9 เท่ากัน)
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ---------------------------------------------------------
        # ย่อขนาดสำหรับแสดงบนหน้าจอ (Window Scaling)
        # ---------------------------------------------------------
        # ภาพต่อกันแนวนอนตอนนี้มีขนาด 3840 x 1080 พิกเซล ซึ่งใหญ่กว่าจอคอมพิวเตอร์ทั่วไป
        # เราจึงต้องย่อ "เฉพาะตอนแสดงหน้าต่าง" ลง 50% ให้เหลือ 1920 x 540
        # (ข้อมูลดิบในตัวแปร color_frame และ depth_frame ยังคงเป็นความละเอียดสูงสุด)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Resolution', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27) เพื่อออกจากลูป
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 2_PostProcessFilter

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ---------------------------------------------------------
# ตั้งค่าความละเอียดสูงสุดของกล้อง D415
# ---------------------------------------------------------
# RGB: 1920 x 1080 ที่ 30 FPS
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
# Depth: 1280 x 720 ที่ 30 FPS
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
print("⚠️ (ตรวจสอบให้แน่ใจว่าเสียบสายเข้ากับพอร์ต USB 3.0)")

# 2. เริ่มต้นการเชื่อมต่อ
profile = pipeline.start(config)

# 3. ตั้งค่าเปิด IR Projector
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ---------------------------------------------------------
# 4. ประกาศตัวแปรสำหรับ Post-Processing Filters
# ---------------------------------------------------------
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()

# ตัวช่วยสำหรับใส่สีให้ภาพ Depth
colorizer = rs.colorizer()

print("กำลังสตรีมภาพความละเอียดสูงสุด พร้อมเปิดใช้งาน Filters...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมล่าสุดจากกล้อง
        frames = pipeline.wait_for_frames()

        # ดึงเฟรมแต่ละประเภท
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # ---------------------------------------------------------
        # 5. นำภาพ Depth ผ่านตัวกรอง (Apply Filters)
        # ---------------------------------------------------------
        # เรียงลำดับ: Spatial (เกลี่ยพิกเซล) -> Temporal (ลดกระพริบ) -> Hole Filling (ถมดำ)
        filtered_depth = spatial_filter.process(depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # ---------------------------------------------------------
        # แปลงข้อมูลเฟรมเป็น Numpy Array
        # ---------------------------------------------------------
        # ภาพ RGB (ขนาด 1920 x 1080)
        color_image = np.asanyarray(color_frame.get_data())
        
        # ภาพ Depth ที่ผ่านการทำ Filter แล้ว (ขนาด 1280 x 720)
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (0, 255, 0)
        thickness = 3 
        
        cv2.putText(color_image, "RGB (1920x1080)", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "Filtered Depth (1280x720)", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # ปรับขนาดและจัด Layout
        # ---------------------------------------------------------
        # ขยายภาพ Depth จาก 720p เป็น 1080p เพื่อให้ความสูงเท่ากับภาพ RGB
        depth_color_resized = cv2.resize(depth_color_image, (color_image.shape[1], color_image.shape[0]))

        # นำภาพมาต่อกันแนวนอน (ซ้าย: RGB | ขวา: Filtered Depth)
        images_hstack = np.hstack((color_image, depth_color_resized))

        # ย่อขนาดลง 50% ให้เหลือ 1920 x 540 เพื่อให้แสดงผลบนหน้าจอคอมพิวเตอร์ได้พอดี
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.5, fy=0.5)

        # แสดงหน้าต่าง OpenCV
        cv2.imshow('Intel RealSense - Max Res & Filters', display_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 3_1_ShowDepth

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดสูงสุด (RGB 1080p, Depth 720p)
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลัง
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตัวจัดตำแหน่งภาพดึงภาพ Depth ไปหาภาพสี (Color)
align_to = rs.stream.color
align = rs.align(align_to)

# ตั้งค่า Filters และ Colorizer
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

print("กำลังสตรีมภาพเปรียบเทียบ Before / After Alignment...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        # รอรับชุดข้อมูลเฟรมดั้งเดิม
        frames = pipeline.wait_for_frames()

        # [ส่วนที่ 1] ดึงภาพ Depth ดั้งเดิม (Raw Depth ก่อนผ่านกระบวนการ Align)
        raw_depth_frame = frames.get_depth_frame()

        # [ส่วนที่ 2] นำเฟรมไปผ่านกระบวนการจัดตำแหน่ง (Alignment Process)
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame or not raw_depth_frame:
            continue

        # นำภาพ Aligned Depth ไปผ่าน Filters เพื่อให้ภาพเนียนขึ้น
        filtered_aligned_depth = spatial_filter.process(aligned_depth_frame)
        filtered_aligned_depth = temporal_filter.process(filtered_aligned_depth)
        filtered_aligned_depth = hole_filling.process(filtered_aligned_depth)

        # ---------------------------------------------------------
        # แปลงเป็น Numpy Array และใส่สีภาพความลึก
        # ---------------------------------------------------------
        color_image = np.asanyarray(color_frame.get_data())
        
        # ใส่สีให้ Raw Depth (ภาพนี้ยังอยู่ที่ขนาด 1280x720 และมุมมองเดิม)
        raw_depth_image = np.asanyarray(colorizer.colorize(raw_depth_frame).get_data())
        
        # ใส่สีให้ Aligned Depth (ภาพนี้ถูกแปลงเป็น 1920x1080 และปรับมุมมองแล้ว)
        aligned_depth_image = np.asanyarray(colorizer.colorize(filtered_aligned_depth).get_data())

        # สร้างภาพ Overlay
        overlay_image = cv2.addWeighted(color_image, 0.5, aligned_depth_image, 0.5, 0)

        # ---------------------------------------------------------
        # ปรับขนาดภาพเพื่อให้ต่อกันเป็นตารางได้
        # ---------------------------------------------------------
        # เนื่องจากภาพ Raw Depth ดั้งเดิมเป็น 720p แต่ภาพอื่นเป็น 1080p 
        # เราต้องขยายขนาดของ Raw Depth ให้สูงเท่าภาพอื่นก่อนนำมาสแตกกัน
        raw_depth_resized = cv2.resize(raw_depth_image, (color_image.shape[1], color_image.shape[0]))

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (255, 255, 255)
        thickness = 3 
        
        cv2.putText(color_image, "1. RGB (Reference)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(raw_depth_resized, "2. Raw Depth (Before Align - Shipped Position)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(aligned_depth_image, "3. Aligned Depth (After Align)", (20, 50), font, 1.3, color_label, thickness)
        cv2.putText(overlay_image, "4. Depth Overlay (Perfect Match)", (20, 50), font, 1.3, color_label, thickness)

        # ---------------------------------------------------------
        # จัดตารางหน้าจอแสดงผล 2x2
        # ---------------------------------------------------------
        top_row = np.hstack((color_image, raw_depth_resized))
        bottom_row = np.hstack((aligned_depth_image, overlay_image))
        images_grid = np.vstack((top_row, bottom_row))

        # ย่อขนาดตารางรวมลง 40% เพื่อให้แสดงผลบนหน้าจอคอมพิวเตอร์ได้โดยไม่ล้น
        display_image = cv2.resize(images_grid, (0, 0), fx=0.4, fy=0.4)

        cv2.imshow('Intel RealSense - Alignment Comparison (2x2)', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

3_AligDepthOverlay

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# 1. สร้าง Pipeline และ Config
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดสูงสุด
config.enable_stream(rs.stream.color, 1920, 1080, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# เปิด IR Projector เบื้องหลังเพื่อให้ Depth แม่นยำ
depth_sensor = profile.get_device().first_depth_sensor()
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ---------------------------------------------------------
# 2. ตั้งค่า Alignment และ Filters
# ---------------------------------------------------------
# สร้างออบเจกต์สำหรับดัดภาพ Depth ให้ตรงกับมุมมองของภาพสี (Color)
align_to = rs.stream.color
align = rs.align(align_to)

spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

print("กำลังสตรีมภาพ (RGB | Aligned Depth | Overlay)...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # ---------------------------------------------------------
        # 3. จัดตำแหน่งภาพ (Align) ก่อนดึงเฟรม
        # ---------------------------------------------------------
        # นำเฟรมทั้งหมดไปผ่านกระบวนการ Align ภาพ Depth จะถูกขยายและดัดให้เท่ากับ RGB (1080p) ทันที
        aligned_frames = align.process(frames)

        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # ---------------------------------------------------------
        # 4. นำภาพ Depth ที่จัดตำแหน่งแล้วไปผ่าน Filters
        # ---------------------------------------------------------
        filtered_depth = spatial_filter.process(aligned_depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # ---------------------------------------------------------
        # 5. แปลงเป็น Numpy Array
        # ---------------------------------------------------------
        color_image = np.asanyarray(color_frame.get_data())
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data())

        # สร้างภาพส่วนที่ 3 (Overlay) โดยการนำภาพ RGB กับ Depth มาผสมกันอย่างละ 50%
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)

        # ---------------------------------------------------------
        # ใส่ข้อความกำกับ (Labels)
        # ---------------------------------------------------------
        font = cv2.FONT_HERSHEY_SIMPLEX
        color_label = (255, 255, 255) # ตัวอักษรสีขาว
        thickness = 3 
        
        cv2.putText(color_image, "1. RGB", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(depth_color_image, "2. Aligned Depth", (20, 50), font, 1.5, color_label, thickness)
        cv2.putText(overlay_image, "3. Depth Overlay", (20, 50), font, 1.5, color_label, thickness)

        # ---------------------------------------------------------
        # นำภาพมาต่อกันแนวนอน 3 ภาพ (ขนาด 1920x1080 ต่อกัน 3 ภาพ)
        # ---------------------------------------------------------
        images_hstack = np.hstack((color_image, depth_color_image, overlay_image))

        # ย่อขนาดลงเหลือ 33% เพื่อให้ความกว้างรวมพอดีกับหน้าจอคอมพิวเตอร์ทั่วไป (ประมาณ 1900px)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.33, fy=0.33)

        cv2.imshow('Intel RealSense - Alignment & Overlay', display_image)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 4_DepthMeasure

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตัวแปร Global สำหรับควบคุมกล่องเมาส์ (Mouse Callback)
# =========================================================
box_size = 80       # ขนาดความกว้าง/ยาวของกล่อง (พิกเซล)
box_x, box_y = 640, 360  # ตำแหน่งเริ่มต้น (กึ่งกลางหน้าจอ 1280x720)
is_dragging = False # สถานะการคลิกค้าง

def mouse_handler(event, x, y, flags, param):
    global box_x, box_y, is_dragging
    
    # เมื่อคลิกเมาส์ซ้าย -> ย้ายกล่องมาที่เคอร์เซอร์และเริ่มการลาก
    if event == cv2.EVENT_LBUTTONDOWN:
        is_dragging = True
        box_x, box_y = x, y
        
    # เมื่อขยับเมาส์ขณะคลิกค้าง -> อัปเดตตำแหน่งกล่อง
    elif event == cv2.EVENT_MOUSEMOVE:
        if is_dragging:
            box_x, box_y = x, y
            
    # เมื่อปล่อยคลิกซ้าย -> หยุดการลาก
    elif event == cv2.EVENT_LBUTTONUP:
        is_dragging = False

# =========================================================
# 2. ตั้งค่าการเชื่อมต่อกล้อง RealSense
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ตั้งค่าความละเอียดที่ 1280x720 ทั้งคู่ เพื่อให้ไม่ต้องย่อภาพและเมาส์ชี้ได้แม่นยำ
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# ดึงค่า Depth Scale ของกล้อง (ค่าตัวคูณเพื่อแปลงข้อมูลดิบเป็นหน่วย 'เมตร')
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# เปิด IR Projector เบื้องหลัง
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 1) 

# ตั้งค่า Alignment และ Filters
align = rs.align(rs.stream.color)
spatial_filter = rs.spatial_filter()
temporal_filter = rs.temporal_filter()
hole_filling = rs.hole_filling_filter()
colorizer = rs.colorizer()

# =========================================================
# 3. เตรียมหน้าต่างแสดงผลและผูกเมาส์เข้ากับหน้าต่าง
# =========================================================
window_name = 'Depth Overlay & Measurement (Click & Drag)'
cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE)
cv2.setMouseCallback(window_name, mouse_handler)

print("กำลังสตรีมภาพ...")
print("🎯 คุณสามารถ 'คลิกและลากเมาส์' บนหน้าต่างเพื่อเลื่อนกล่องวัดระยะได้")
print("⚠️ กดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # จัดตำแหน่ง (Align) ให้พิกเซลตรงกัน
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # นำภาพ Depth ผ่าน Filters เพื่อความเนียน
        filtered_depth = spatial_filter.process(aligned_depth_frame)
        filtered_depth = temporal_filter.process(filtered_depth)
        filtered_depth = hole_filling.process(filtered_depth)

        # แปลงข้อมูลเป็น Numpy Array
        color_image = np.asanyarray(color_frame.get_data())
        depth_data_array = np.asanyarray(filtered_depth.get_data()) # ค่าความลึกดิบสำหรับคำนวณ
        depth_color_image = np.asanyarray(colorizer.colorize(filtered_depth).get_data()) # ภาพสีความลึกสำหรับโชว์

        # สร้างภาพ Overlay (ผสม RGB 50% + Depth 50%)
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)

        # =========================================================
        # 4. คำนวณระยะทางภายในกล่อง (ROI Distance Calculation)
        # =========================================================
        # ป้องกันไม่ให้ขอบเขตกล่องทะลุออกนอกภาพ (Clamp boundaries)
        y1 = max(0, box_y - box_size // 2)
        y2 = min(720, box_y + box_size // 2)
        x1 = max(0, box_x - box_size // 2)
        x2 = min(1280, box_x + box_size // 2)

        # ดึงชุดข้อมูลความลึกเฉพาะพื้นที่ในกล่อง (ROI)
        roi_depth = depth_data_array[y1:y2, x1:x2]

        # แปลงข้อมูลดิบเป็นหน่วย 'เมตร'
        distances = roi_depth * depth_scale

        # กรองเอาเฉพาะค่าที่มีความลึก (มากกว่า 0) เพื่อนำมาหาค่าเฉลี่ย
        valid_distances = distances[distances > 0]
        
        if len(valid_distances) > 0:
            avg_distance = np.mean(valid_distances)
            text_distance = f"Dist: {avg_distance:.3f} m"
            color_box = (0, 255, 0) # สีเขียวเมื่อจับระยะได้
        else:
            text_distance = "Dist: N/A"
            color_box = (0, 0, 255) # สีแดงเมื่อเป็นจุดบอด (หาความลึกไม่ได้)

        # =========================================================
        # 5. วาดกล่องและตัวหนังสือลงบนภาพ
        # =========================================================
        # วาดกรอบสี่เหลี่ยม
        cv2.rectangle(overlay_image, (x1, y1), (x2, y2), color_box, 2)
        
        # วาดจุดกึ่งกลาง (Center dot)
        cv2.circle(overlay_image, (box_x, box_y), 4, color_box, -1)

        # ใส่ข้อความบอกระยะทางไว้บนกล่อง
        cv2.putText(overlay_image, text_distance, (x1, y1 - 10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)
        
        # ใส่ข้อความแนะนำวิธีใช้ที่มุมซ้ายบน
        cv2.putText(overlay_image, "Click & drag the box to measure distance", 
                    (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # =========================================================
        # แสดงผลภาพ
        # =========================================================
        cv2.imshow(window_name, overlay_image)

        # ตรวจสอบการกดปุ่ม 'q' หรือ 'ESC' (27)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    # คืนทรัพยากรกล้อง
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

# 5_DepthClipping

In [2]:
import pyrealsense2 as rs
import numpy as np
import cv2

# =========================================================
# 1. ตั้งค่าการเชื่อมต่อกล้อง
# =========================================================
pipeline = rs.pipeline()
config = rs.config()

# ใช้ความละเอียด 1280x720 เพื่อให้สัดส่วนภาพเท่ากันและทำงานได้ลื่นไหล
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
config.enable_stream(rs.stream.depth, 1280, 720, rs.format.z16, 30)

print("กำลังเชื่อมต่อกล้อง Intel RealSense...")
profile = pipeline.start(config)

# ดึงค่า Depth Scale (ตัวคูณเพื่อแปลงค่าดิบให้เป็นหน่วยเมตร)
depth_sensor = profile.get_device().first_depth_sensor()
depth_scale = depth_sensor.get_depth_scale()

# เปิด IR Projector เบื้องหลัง
if depth_sensor.supports(rs.option.emitter_enabled):
    depth_sensor.set_option(rs.option.emitter_enabled, 0) 

# ตั้งค่า Alignment และ ตัวใส่สี
align = rs.align(rs.stream.color)
colorizer = rs.colorizer()

# =========================================================
# 2. ตั้งค่าระยะทางที่ต้องการตัด (Depth Clipping)
# =========================================================
clipping_distance_in_meters = 3.5 # กำหนดระยะที่ต้องการถมดำ (เมตร)

# แปลงหน่วย 'เมตร' ให้กลับเป็น 'ค่าดิบ (Raw Depth)' ที่กล้องเข้าใจ เพื่อให้ประมวลผลได้ไวขึ้น
clipping_distance = clipping_distance_in_meters / depth_scale

print(f"กำลังสตรีมภาพ (ตั้งค่าตัดฉากหลังที่ระยะ > {clipping_distance_in_meters} เมตร)...")
print("⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม")

try:
    while True:
        frames = pipeline.wait_for_frames()

        # จัดตำแหน่ง (Align) ให้พิกเซลความลึกและภาพสีตรงกันเป๊ะ
        aligned_frames = align.process(frames)
        color_frame = aligned_frames.get_color_frame()
        aligned_depth_frame = aligned_frames.get_depth_frame()

        if not color_frame or not aligned_depth_frame:
            continue

        # =========================================================
        # 3. แปลงเป็น Numpy Array
        # =========================================================
        color_image = np.asanyarray(color_frame.get_data())
        depth_data = np.asanyarray(aligned_depth_frame.get_data()) # ค่าความลึกแบบดิบ
        depth_color_image = np.asanyarray(colorizer.colorize(aligned_depth_frame).get_data()) # ความลึกแบบมีสี

        # =========================================================
        # 4. ประมวลผลภาพทั้ง 3 ส่วน
        # =========================================================
        
        # [ส่วนที่ 1] ภาพ RGB ดั้งเดิม (มีอยู่ในตัวแปร color_image แล้ว)
        
        # [ส่วนที่ 2] ภาพ Overlay (RGB ผสมกับ Depth)
        overlay_image = cv2.addWeighted(color_image, 0.5, depth_color_image, 0.5, 0)
        
        # [ส่วนที่ 3] ภาพที่ถมดำเมื่อเกิน 3.5 เมตร (Background Removed)
        # เนื่องจากภาพสีมี 3 ช่องสี (BGR) แต่ภาพ Depth มีแค่ 1 มิติ
        # เราจึงต้องก็อปปี้มิติของ Depth ให้ซ้อนกัน 3 ชั้นก่อน (เพื่อใช้เป็นเงื่อนไข)
        depth_image_3d = np.dstack((depth_data, depth_data, depth_data)) 

        # สร้างภาพใหม่ โดยบอกว่า: 
        # ถ้าค่าความลึก > 3.5 เมตร หรือ หาค่าไม่ได้ (<= 0) ให้แทนที่ด้วยสีดำ (0)
        # ถ้าระยะไม่เกิน 3.5 เมตร ให้ใช้พิกเซลสีจาก color_image เหมือนเดิม
        bg_removed_image = np.where((depth_image_3d > clipping_distance) | (depth_image_3d <= 0), 0, color_image)

        # =========================================================
        # 5. ใส่ข้อความกำกับ (Labels)
        # =========================================================
        font = cv2.FONT_HERSHEY_SIMPLEX
        cv2.putText(color_image, "1. RGB", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(overlay_image, "2. Overlay", (20, 50), font, 1.2, (255, 255, 255), 2)
        cv2.putText(bg_removed_image, f"3. Clipped (< {clipping_distance_in_meters}m)", (20, 50), font, 1.2, (255, 255, 255), 2)

        # =========================================================
        # 6. จัด Layout และแสดงผล
        # =========================================================
        # นำภาพมาต่อกันแนวนอน 3 ภาพ
        images_hstack = np.hstack((color_image, overlay_image, bg_removed_image))

        # ย่อขนาดลง 40% (เหลือ 60%) เพื่อไม่ให้ล้นหน้าจอ (จาก 3840px จะเหลือประมาณ 1500px)
        display_image = cv2.resize(images_hstack, (0, 0), fx=0.4, fy=0.4)

        cv2.imshow('Intel RealSense - 3 Layers (Clipping)', display_image)

        # กด 'q' เพื่อออก
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    cv2.waitKey(1) 
    print("ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว")

กำลังเชื่อมต่อกล้อง Intel RealSense...
กำลังสตรีมภาพ (ตั้งค่าตัดฉากหลังที่ระยะ > 3.5 เมตร)...
⚠️ คลิกที่หน้าต่างวิดีโอ แล้วกดปุ่ม 'q' หรือ 'ESC' เพื่อปิดโปรแกรม
ปิดการสตรีมและเคลียร์หน้าต่างเรียบร้อยแล้ว
